In [22]:
!pip install opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   - -------------------------------------- 1.6/40.2 MB 7.4 MB/s eta 0:00:06
   --- ------------------------------------ 3.1/40.2 MB 7.3 MB/s eta 0:00:06
   ---- ----------------------------------- 4.7/40.2 MB 7.3 MB/s eta 0:00:05
   ----- ---------------------------------- 6.0/40.2 MB 7.3 MB/s eta 0:00:05
   ------- -------------------------------- 7.6/40.2 MB 7.3 MB/s eta 0:00:05
   --------- ------------------------------ 9.2/40.2 MB 7.3 MB/s eta 0:00:05
   ---------- ----------------------------- 10.7/40.2 MB 7.2 MB/s eta 0:00:05
   ------------ --------------------------- 12.3/40.2 MB 7.2 MB/s eta 0:00:04
   ------------- -------------------------- 13.9/40.2 MB 7.3 MB/s eta 0:00:04
   -------------- ------------------------- 14.9/40.2 MB 7.2 MB/s eta 0:00:04
   --------------- ------------------------ 16.0/40.2 MB 6.9 MB/s eta 0:00:04
   ----------------- ---------------------- 17.8/40.2 MB 6.9 MB/s eta 0:00:04


In [60]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow import keras
import cv2

<font size="5">Prepare the dataset for model training</font>

In [44]:
#Pad images to same size
def pad_image(img, target_size=(256, 256)):
    h, w = img.shape[:2]
    th, tw = target_size

    # If the image is larger than target, resize down first
    if h > th or w > tw:
        img = cv2.resize(img, (tw, th))
        h, w = img.shape[:2]

    # Create blank canvas
    canvas = np.zeros((th, tw, 3), dtype=np.uint8)

    # Compute offsets for centering
    y_offset = (th - h) // 2
    x_offset = (tw - w) // 2

    # Place the image on the canvas
    canvas[y_offset:y_offset+h, x_offset:x_offset+w] = img
    return canvas

In [47]:
labels_df = pd.read_csv('labels.csv',sep=',', header=None)
labels_df.columns = ['images_id','class','x_min','y_min','x_max','y_max']

labels_df['images_id'] = labels_df['images_id'].apply(lambda x: f"{x:08d}")
labels_df =labels_df.iloc[:1000]

images_dir = 'images/'
images = []

for index,row in labels_df.iterrows():
    images_path = os.path.join(images_dir, f"{row['images_id']}.jpg")
    img = cv2.imread(images_path)
    padded_img = pad_image(img, target_size=(256, 256))
    if padded_img is not None:
        images.append(padded_img)
    else:
        print(f"Error loading image: {images_path}")

images = np.stack(images,axis=0)
if len(images) == 0:
    print("No images loaded.Please check the image path")
else:
    print(f"{len(images)} images loaded succesfully")

1000 images loaded succesfully


In [50]:
#Analyze the distribution of vehicle types
vehicle_types = labels_df['class'].value_counts()
print("Distribution of vehicle types:")
print(vehicle_types)

#Sorting the image filenames
labels_df = labels_df.sort_values('images_id')

Distribution of vehicle types:
class
car                      682
pickup_truck             111
motorized_vehicle         61
articulated_truck         30
work_van                  29
bus                       28
pedestrian                23
single_unit_truck         18
bicycle                   12
non-motorized_vehicle      5
motorcycle                 1
Name: count, dtype: int64


In [53]:
if len(images) > 0:
    processed_images = [cv2.resize(img, (224,224)) for img in images]
    processed_images = np.array(processed_images)
    print("Images resized successfully")
                        

Images resized successfully


In [55]:
labels = labels_df['class'].to_numpy()
bounding_boxes = labels_df[['x_min','y_min','x_max','y_max']].to_numpy()

#Convert labels to one-hot encoding
unique_labels = np.unique(labels)
label_to_index = {label:index for index,label in enumerate(unique_labels)}
index_to_label = {index:label for index,label in enumerate(unique_labels)}
labels = np.array([label_to_index[label] for label in labels])


In [56]:
X_train, X_test, y_train,y_test, bbox_train, bbox_test = train_test_split(processed_images,labels,bounding_boxes,test_size=2,random_state=42)

<font size="5">Create CNN architechture for object detection</font>

In [62]:
def create_model(input_shape,num_classes):
    inputs=keras.Input(shape=input_shape)
    x = layers.Conv2D(32,(3,3),activation="relu")(inputs)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64,(3,3),activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64,(3,3),activation="relu")(x)
    x = layers.Flatten()(x)
    vehicle_class = layers.Dense(num_classes,activation="softmax", name='vehicle_class')(x)
    bounding_box = layers.Dense(4, name='bounding_box')(x)

    model = keras.Model(inputs=inputs,outputs=[vehicle_class, bounding_box])
    return model

input_shape=processed_images[0].shape
num_classes = len(unique_labels)
model=create_model(input_shape,num_classes)

model.compile(optimizer='adam',
             loss={'vehicle_class':'sparse_categorical_crossentropy','bounding_box':'mse'},
             metrics={'vehicle_class':'accuracy','bounding_box':'mae'})

model.fit(X_train,{'vehicle_class':y_train, 'bounding_box':bbox_train},epochs=20,validation_data=(X_test,{'vehicle_class':y_test,'bounding_box':bbox_test}))
    

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 26s 675ms/step - bounding_box_loss: 220788.2031 - bounding_box_mae: 263.2578 - loss: 225906.6562 - vehicle_class_accuracy: 0.4138 - vehicle_class_loss: 163.8813 - val_bounding_box_loss: 40964.4492 - val_bounding_box_mae: 168.5181 - val_loss: 40964.4492 - val_vehicle_class_accuracy: 1.0000 - val_vehicle_class_loss: 0.0000e+00
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 611ms/step - bounding_box_loss: 25743.8535 - bounding_box_mae: 123.8702 - loss: 25771.3848 - vehicle_class_accuracy: 0.4900 - vehicle_class_loss: 78.3041 - val_bounding_box_loss: 46340.4922 - val_bounding_box_mae: 178.3340 - val_loss: 46340.4922 - val_vehicle_class_accuracy: 1.0000 - val_vehicle_class_loss: 0.0000e+00
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 18s 564ms/step - bounding_box_loss: 22258.1445 - bounding_box_mae: 115.7071 - loss: 22634.1250 - vehicle_class_accuracy: 0.5170 - vehicle_class_loss: 78.1745 - val_bounding_box_loss: 37571.8750 - val_bounding_box_mae: 168.5920 - val_lo